In [1]:
import os, gc, random, re
import torch
from torch.utils.data import DataLoader
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm import tqdm
import json

In [7]:
file_path = "MMLU_data.json"

with open(file_path, 'r', encoding='utf-8') as f:
    combined_data = json.load(f)

updated_input = [item[0] for item in combined_data]
question = [item[1] for item in combined_data]
subject = [item[2] for item in combined_data]
choices = [item[3] for item in combined_data]
answer = [item[4] for item in combined_data]

In [8]:
model_name = "./models/Meta-Llama-3-8B-Instruct"
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
access_token = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

device = next(model.parameters()).device

You passed `quantization_config` to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` attribute will be overwritten with the one you passed to `from_pretrained`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of the model checkpoint at /home/yang/Desktop/Doni/New Jailbreak Project/PAT/models/Meta-Llama-3-8B-Instruct were not used when initializing LlamaForCausalLM: ['model.layers.0.mlp.down_proj.weight_format', 'model.layers.0.mlp.gate_proj.weight_format', 'model.layers.0.mlp.up_proj.weight_format', 'model.layers.0.self_attn.k_proj.weight_format', 'model.layers.0.self_attn.o_proj.weight_format', 'model.layers.0.self_attn.q_proj.weight_format', 'model.layers.0.self_attn.v_proj.weight_format', 'model.layers.1.mlp.down_proj.weight_format', 'model.layers.1.mlp.gate_proj.weight_format', 'model.layers.1.mlp.up_proj.weight_format', 'model.layers.1.self_attn.k_proj.weight_format', 'model.layers.1.self_attn.o_proj.weight_format', 'model.layers.1.self_attn.q_proj.weight_format', 'model.layers.1.self_attn.v_proj.weight_format', 'model.layers.10.mlp.down_proj.weight_format', 'model.layers.10.mlp.gate_proj.weight_format', 'model.layers.10.mlp.up_proj.weight_format', 'model.layers.10.self_at

In [9]:
def format_prompt(prompts, system_prompt=""):
    formatted_prompts = []
    answer = "The correct answer is"
    
    for prompt in prompts:
        if "vicuna" in model_name.lower():
            if system_prompt != "":
                formatted_prompt = f"{system_prompt}\n\nUSER: {prompt}\nASSISTANT: {answer}"
            else:
                formatted_prompt = f"USER: {prompt}\nASSISTANT: {answer}"
        elif "llama-3" in model_name.lower():
            formatted_prompt = (
                f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
                f"{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
                f"{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{answer}")
        elif "llama-2" in model_name.lower() or "mistral" in model_name.lower():
            formatted_prompt = f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{prompt} [/INST]\n{answer}"
        elif "deepseek" in model_name.lower():
            if system_prompt != "":
                formatted_prompt = f"User: {system_prompt} {prompt}\nAssistant: {answer}"
            else:
                formatted_prompt = f"User: {prompt}\nAssistant: {answer}"
        elif "openchat" in model_name.lower():
            formatted_prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{prompt}\n<|assistant|>\n{answer}"
        else:
            print("A chat template for this model is not defined. Add the chat template in the format_prompt function.")
            return None
        formatted_prompts.append(formatted_prompt)
    
    return formatted_prompts

In [10]:
def generate_text(prompts, batch_size=64):
    generated_texts = []
    dataloader = DataLoader(prompts, batch_size=batch_size, shuffle=False)

    for batch in tqdm(dataloader, desc="Generating responses", total=len(prompts)//batch_size + 1):
        # Tokenize the batch
        tokenized_inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        # Generate
        with torch.no_grad():
            generated_ids = model.generate(input_ids=tokenized_inputs["input_ids"],
                                           attention_mask=tokenized_inputs["attention_mask"],
                                           max_new_tokens=64, pad_token_id=tokenizer.pad_token_id,
                                           do_sample=False, repetition_penalty=1)

        # Decode
        for i in range(len(batch)):
            output_tokens = generated_ids[i][tokenized_inputs["input_ids"].size(1):].cpu()
            generated_text = tokenizer.decode(output_tokens, skip_special_tokens=True)
            generated_texts.append(generated_text)

        # Clear GPU memory
        del tokenized_inputs, generated_ids, output_tokens, generated_text
        torch.cuda.empty_cache()
        gc.collect()

    return generated_texts

In [11]:
results = []

format_clean_prompts = format_prompt(updated_input)

# Process clean prompts
clean_outputs = generate_text(format_clean_prompts)
for prompt, output in zip(updated_input, clean_outputs):
    results.append({"model": model_name, "input": prompt, "output": output})

# Clear GPU memory
del model, tokenizer, format_clean_prompts
torch.cuda.empty_cache()
gc.collect()

df_inference = pd.DataFrame(results)
df_inference["subject"] = subject
df_inference["answer"] = answer

# Save to CSV
df_inference.to_csv("evaluation_llama3_MMLU.csv", index=False)
print("Processing complete. Results saved to evaluation_llama3_MMLU.csv.")

Generating responses:   0%|                                                                                             | 0/22 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/home/yang/.conda/envs/DRO/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Generating responses: 100%|████████████████████████████████████████████████████████████████████████████████████| 22/22 [03:57<00:00, 10.82s/it]

Processing complete. Results saved to evaluation_llama3_MMLU.csv.


In [12]:
import pandas as pd
df = pd.read_csv('evaluation_llama3_MMLU.csv')
pd.set_option('display.max_colwidth', None)

In [13]:
def extract_answer_index(text):
    match = re.search(r"(?:^|(?<=\W))\s*([A-D])\s*(?:$|(?=\W))", text)
    if match:
        letter = match.group(1)
        return {"A": 0, "B": 1, "C": 2, "D": 3}[letter]
    return -1

df["output"] = df["output"].apply(lambda x: x if isinstance(x, str) else "")
df["predicted_answer"] = df["output"].apply(extract_answer_index)

In [14]:
df["correct"] = df["predicted_answer"] == df["answer"]
accuracy = df["correct"].mean()

print(f"Accuracy on {len(df)} examples: {accuracy:.2%}")

Accuracy on 1404 examples: 65.46%
